In [ ]:
!wget -q http://www.statmt.org/europarl/v7/fr-en.tgz
!tar -xzf fr-en.tgz
!pip install -q transformers sentencepiece sacrebleu
print('Done.')

In [ ]:
import sacrebleu

def tokenize(s):
    return s.lower().strip().split()

with open('europarl-v7.fr-en.fr', encoding='utf-8') as f:
    fr_raw = [l.strip() for l in f]
with open('europarl-v7.fr-en.en', encoding='utf-8') as f:
    en_raw = [l.strip() for l in f]

pairs  = [(f, e) for f, e in zip(fr_raw, en_raw) if f and e]
fr_raw = [p[0] for p in pairs]
en_raw = [p[1] for p in pairs]
n = len(fr_raw)
print(f'Total sentence pairs: {n:,}')

In [ ]:
# Same Koehn-style test split used by the phrase-based system.
TRAIN_SIZE         = 50000
EUROPARL_TEST_SIZE = 1755
TEST_SRC_MIN_LEN   = 5
TEST_SRC_MAX_LEN   = 15

qual_idx     = [i for i in range(n) if TEST_SRC_MIN_LEN <= len(tokenize(fr_raw[i])) <= TEST_SRC_MAX_LEN]
test_idx_set = set(qual_idx[-EUROPARL_TEST_SIZE:])
test_order   = sorted(test_idx_set)

test_fr_raw = [fr_raw[i] for i in test_order]   # original-case French for the tokenizer
test_en_ref = [en_raw[i] for i in test_order]   # English references for BLEU

print(f'Test sentences: {len(test_fr_raw):,}')
print(f'Sample FR: {test_fr_raw[0]}')
print(f'Sample EN: {test_en_ref[0]}')

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
from tqdm import tqdm

model_name = 'Helsinki-NLP/opus-mt-fr-en'
tokenizer  = MarianTokenizer.from_pretrained(model_name)
model      = MarianMTModel.from_pretrained(model_name)
print('Model loaded.')

In [ ]:
def translate_batch(sentences, batch_size=32):
    out = []
    for i in tqdm(range(0, len(sentences), batch_size)):
        batch  = sentences[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='pt', padding=True,
                           truncation=True, max_length=512)
        preds  = model.generate(**inputs)
        out   += [tokenizer.decode(t, skip_special_tokens=True) for t in preds]
    return out

translations = translate_batch(test_fr_raw)

bleu = sacrebleu.corpus_bleu(translations, [test_en_ref])
print(f'\nOPUS-MT Europarl in-domain BLEU ({len(test_fr_raw)} sentences): {bleu.score:.2f}')

print('\nSample translations:')
for i in range(5):
    print(f'FR:   {test_fr_raw[i]}')
    print(f'REF:  {test_en_ref[i]}')
    print(f'OPUS: {translations[i]}')
    print()